
# Ataques Adversariais: Comprometendo Sistemas Baseados em Machine Learning

# Instalar Bibliotecas Necessárias

In [2]:
!pip install textattack

#
# Note: No need to install textattack[tensorflow].
# If the current environment explicitly demands a text embedding framework that uses 
# TensorFlow Hub like Universal Sentence Encoder under the hood, it will prompt you at runtime. 
# We are running a PyTorch pipeline, not a TensorFlow one. 
# Uses: torch, torch.nn, LSTMForClassification (uses PyTorch helper) and wrapping it in PyTorchModelWrapper.

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 17.5 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.7/769.7 kB 22.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.0 MB/s  0:00:00
  Created wheel for pinyin: filename=pinyin-0.4.0-py3-none-any.whl size=3630548 sha256=9c8930aa930e5dad91b1fc101c4463e73bfe57bc4829d8d9eccd8f90fbbe84c6
  Stored in directory: /root/.cache/pip/wheels/45/23/be/670fc418eadc93220baf097682c12d3bd6210e28e080317c9a
  Created wheel for word2number: filename=word2number-1.1-py3-none-any.whl size=5659 sha256=f3ba0003babd740a5945abca053405c7f0e600fe67e1e2dca79804c9e2ec296c
  Stored in directory: /root/.cache/pip/wheels/aa/2b/71/ede4a3c3520a837477

# Importações

In [3]:
import numpy as np
print(np.__version__)

2.4.6


In [4]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from textattack.models.helpers import LSTMForClassification
from textattack.datasets import HuggingFaceDataset
from textattack.models.wrappers.pytorch_model_wrapper import PyTorchModelWrapper
from textattack.attack_recipes import PWWSRen2019, HotFlipEbrahimi2017
from textattack import AttackArgs
from textattack.datasets import Dataset
from textattack import Attacker


import nltk

nltk.download('averaged_perceptron_tagger_eng') # TEM QUE TER ESSE ARQUIVO BAIXADO

textattack: Updating TextAttack package dependencies.
textattack: Downloading NLTK required packages.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw to /root/nltk_data...
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape s

True

### **Modelo Alvo (Victim Model): LSTM para Classificação de Texto**  
Neste notebook, utilizamos um modelo **Long Short-Term Memory (LSTM)** como **modelo alvo** para demonstrar ataques adversariais. LSTMs são um tipo de rede neural recorrente (RNN) especialmente eficaz para dados sequenciais, como texto, pois conseguem capturar dependências de longo prazo e informações contextuais.

#### **Arquitetura do Modelo**  
O modelo alvo é composto por:
1. **Dropout**: Nesse exemplo está sem dropout.  
2. **Camada de Embedding**: Converte palavras/tokens em representações vetoriais densas. Neste caso Utilizaremos o Glove Embeddings.
3. **Camada LSTM**: Processa os dados sequenciais, mantendo o contexto relevante ao longo do tempo.  
4. **Camada Linear**: Produz as previsões finais que neste caso são 2 por se tratar de uma classificação binária.  


In [5]:
# @title
"""
LSTM 4 Classification
---------------------------------------------------------------------
"""

import json
import os

import torch
from torch import nn as nn

import textattack
from textattack.model_args import TEXTATTACK_MODELS
from textattack.models.helpers import (
    GloveEmbeddingLayer,
    LSTMForClassification as TextAttackLSTMForClassification,
)
from textattack.models.helpers.utils import load_cached_state_dict
from textattack.shared import utils


class LSTMForClassification(TextAttackLSTMForClassification):
    """A long short-term memory neural network for text classification.

    We use different versions of this network to pretrain models for
    text classification.
    """

    def __init__(
        self,
        hidden_size=150,
        depth=1,
        dropout=0.3,
        num_labels=2,
        max_seq_length=128,
        model_path=None,
        emb_layer_trainable=True,
    ):
        # Inicializa nn.Module sem executar o construtor original do TextAttack.
        nn.Module.__init__(self)

        self._config = {
            "architectures": "LSTMForClassification",
            "hidden_size": hidden_size,
            "depth": depth,
            "dropout": dropout,
            "num_labels": num_labels,
            "max_seq_length": max_seq_length,
            "model_path": model_path,
            "emb_layer_trainable": emb_layer_trainable,
        }

        if depth <= 1:
            dropout = 0

        self.drop = nn.Dropout(dropout)
        self.emb_layer_trainable = emb_layer_trainable

        self.emb_layer = GloveEmbeddingLayer(
            emb_layer_trainable=emb_layer_trainable
        )

        self.word2id = self.emb_layer.word2id

        self.encoder = nn.LSTM(
            input_size=self.emb_layer.n_d,
            hidden_size=hidden_size // 2,
            num_layers=depth,
            dropout=dropout,
            bidirectional=True,
        )

        d_out = hidden_size
        self.out = nn.Linear(d_out, num_labels)
        self.softmax = nn.Softmax(dim=1)

        self.tokenizer = textattack.models.tokenizers.GloveTokenizer(
            word_id_map=self.word2id,
            unk_token_id=self.emb_layer.oovid,
            pad_token_id=self.emb_layer.padid,
            max_length=max_seq_length,
        )

        if model_path is not None:
            self.load_from_disk(model_path)

        self.eval()

    def load_from_disk(self, model_path):
        import warnings

        warnings.warn(
            "`load_from_disk` method is deprecated. "
            "Please save and load using `save_pretrained` and `from_pretrained` methods.",
            DeprecationWarning,
            stacklevel=2,
        )

        self.load_state_dict(load_cached_state_dict(model_path))
        self.eval()

    def save_pretrained(self, output_path):
        if not os.path.exists(output_path):
            os.makedirs(output_path)

        state_dict = {k: v.cpu() for k, v in self.state_dict().items()}

        torch.save(
            state_dict,
            os.path.join(output_path, "pytorch_model.bin"),
        )

        with open(os.path.join(output_path, "config.json"), "w") as f:
            json.dump(self._config, f)

    @classmethod
    def from_pretrained(cls, name_or_path):
        if name_or_path in TEXTATTACK_MODELS:
            path = utils.download_from_s3(TEXTATTACK_MODELS[name_or_path])
        else:
            path = name_or_path

        config_path = os.path.join(path, "config.json")

        if os.path.exists(config_path):
            with open(config_path, "r") as f:
                config = json.load(f)
        else:
            config = {
                "architectures": "LSTMForClassification",
                "hidden_size": 150,
                "depth": 1,
                "dropout": 0.3,
                "num_labels": 2,
                "max_seq_length": 128,
                "model_path": None,
                "emb_layer_trainable": True,
            }

        del config["architectures"]

        model = cls(**config)
        state_dict = load_cached_state_dict(path)
        model.load_state_dict(state_dict)

        return model

    def forward(self, _input):
        self.encoder.flatten_parameters()

        emb = self.emb_layer(_input.t())
        emb = self.drop(emb)

        output, hidden = self.encoder(emb)
        output = torch.max(output, dim=0)[0]

        output = self.drop(output)
        pred = self.out(output)

        return pred

    def get_input_embeddings(self):
        return self.emb_layer.embedding

In [6]:
model = LSTMForClassification()

textattack: Downloading https://textattack.s3.amazonaws.com/word_embeddings/glove200.
100%|██████████| 389M/389M [00:09<00:00, 40.2MB/s] 
textattack: Unzipping file /root/.cache/textattack/tmpqehjwq3k.zip to /root/.cache/textattack/word_embeddings/glove200.
textattack: Successfully saved word_embeddings/glove200 to cache.


In [7]:
print(model)

LSTMForClassification(
  (drop): Dropout(p=0, inplace=False)
  (emb_layer): GloveEmbeddingLayer(
    (embedding): Embedding(400001, 200)
  )
  (encoder): LSTM(200, 75, bidirectional=True)
  (out): Linear(in_features=150, out_features=2, bias=True)
  (softmax): Softmax(dim=1)
)


### **Classe `TextDataset`**  
Esta classe é um **Dataset customizado do PyTorch** que prepara textos e rótulos para treinamento/avaliação do modelo LSTM. Ela:  

1. **Tokeniza textos**: Converte frases em sequências de IDs (compatíveis com a camada de embedding).  
2. **Processa rótulos**: Transforma labels em tensores do PyTorch.  
3. **Integra com o DataLoader**: Permite criar batches e embaralhar os dados automaticamente.

In [8]:
class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.tokenizer.encode(self.texts[idx])
        input_tensor = torch.tensor(tokens, dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return input_tensor, label

### **Dataset IMDB**  
O **IMDB** é um dataset utilizado para análise de sentimentos, contendo 50k avaliações de filmes rotuladas como positivas (1) ou negativas (0). Aqui utilizaremos ele diretamente da plataforma huggingface

In [9]:
splits = {
    'train': 'plain_text/train-00000-of-00001.parquet',
    'test': 'plain_text/test-00000-of-00001.parquet',
    'unsupervised': 'plain_text/unsupervised-00000-of-00001.parquet'
}

In [10]:
df = pd.read_parquet(
    "hf://datasets/stanfordnlp/imdb/" + splits["train"]
    )

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [11]:
df.shape

(25000, 2)

In [12]:
df.head()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


### **Treinamento do Modelo**
Pipeline básico para treinar o LSTM no dataset IMDB:

1. **Pré-processamento**:
   - `TextDataset` converte textos para tensores
   - `DataLoader` cria batches com padding

2. **Configuração**:
   - Loss: `CrossEntropyLoss`
   - Otimizador: `Adam` (lr=0.001)
   - Execução na GPU

3. **Loop principal**:
   - Apenas 5 épocas por questôes de tempo já que o foco são os ataques
   - Forward/backward pass
   - Atualização dos pesos
   - Log da loss por época

In [13]:
# Criar Dataset e DataLoader
dataset = TextDataset(df, model.tokenizer)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=lambda batch: tuple(zip(*batch)))

In [14]:
# Definir função de perda e otimizador
import tqdm 


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Enviar modelo para GPU se disponível
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Treinamento
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in tqdm.tqdm(dataloader):
        inputs, labels = batch
        inputs = torch.nn.utils.rnn.pad_sequence(inputs, batch_first=True).to(device)  # Padding para garantir tamanho igual
        labels = torch.tensor(labels, dtype=torch.long).to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Época {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}")


100%|██████████| 782/782 [00:37<00:00, 20.81it/s]


Época 1/10, Loss: 358.6767


100%|██████████| 782/782 [00:37<00:00, 20.98it/s]


Época 2/10, Loss: 193.0758


100%|██████████| 782/782 [00:37<00:00, 20.94it/s]


Época 3/10, Loss: 79.1286


... (10 outputs left out)

100%|██████████| 782/782 [00:37<00:00, 20.99it/s]


Época 9/10, Loss: 0.0667


100%|██████████| 782/782 [00:37<00:00, 20.79it/s]

Época 10/10, Loss: 0.0349


In [17]:
# Salvar o modelo treinado
model.save_pretrained("lstm_imdb_model")

### **Wrapper**

Antes da realização do ataque a bibliteca textattack necessita que o modelo vítima esteja encapsulado em um wrapper para padronizar as entradas e saidas do modelo para que fique compativel com a implementação dos ataques da biblioteca.


In [19]:
# carregar o modelo salvo
loaded_model = LSTMForClassification.from_pretrained("lstm_imdb_model")
model = loaded_model.to(device)

In [20]:
model_wrapped = PyTorchModelWrapper(model, model.tokenizer)

In [21]:
# @title
"""
PyTorch Model Wrapper
--------------------------
"""

import torch
from torch.nn import CrossEntropyLoss

import textattack

from textattack.models.wrappers import ModelWrapper

torch.cuda.empty_cache()


class PyTorchModelWrapper(ModelWrapper):
    """Loads a PyTorch model (`nn.Module`) and tokenizer.

    Args:
        model (torch.nn.Module): PyTorch model
        tokenizer: tokenizer whose output can be packed as a tensor and passed to the model.
            No type requirement, but most have `tokenizer` method that accepts list of strings.
    """

    def __init__(self, model, tokenizer):
        if not isinstance(model, torch.nn.Module):
            raise TypeError(
                f"PyTorch model must be torch.nn.Module, got type {type(model)}"
            )

        self.model = model
        self.tokenizer = tokenizer

    def to(self, device):
        self.model.to(device)

    def __call__(self, text_input_list, batch_size=32):
        model_device = next(self.model.parameters()).device
        ids = self.tokenizer(text_input_list)
        ids = torch.tensor(ids).to(model_device)

        with torch.no_grad():
            outputs = textattack.shared.utils.batch_model_predict(
                self.model, ids, batch_size=batch_size
            )

        return outputs

    def get_grad(self, text_input, loss_fn=CrossEntropyLoss()):
        """Get gradient of loss with respect to input tokens.

        Args:
            text_input (str): input string
            loss_fn (torch.nn.Module): loss function. Default is `torch.nn.CrossEntropyLoss`
        Returns:
            Dict of ids, tokens, and gradient as numpy array.
        """

        if not hasattr(self.model, "get_input_embeddings"):
            raise AttributeError(
                f"{type(self.model)} must have method `get_input_embeddings` that returns `torch.nn.Embedding` object that represents input embedding layer"
            )
        if not isinstance(loss_fn, torch.nn.Module):
            raise ValueError("Loss function must be of type `torch.nn.Module`.")

        self.model.train()

        embedding_layer = self.model.get_input_embeddings()
        original_state = embedding_layer.weight.requires_grad
        embedding_layer.weight.requires_grad = True

        emb_grads = []

        def grad_hook(module, grad_in, grad_out):
            emb_grads.append(grad_out[0])

        emb_hook = embedding_layer.register_backward_hook(grad_hook)

        self.model.zero_grad()
        model_device = next(self.model.parameters()).device
        ids = self.tokenizer([text_input])
        ids = torch.tensor(ids).to(model_device)

        predictions = self.model(ids)

        output = predictions.argmax(dim=1)
        loss = loss_fn(predictions, output)
        loss.backward()

        # grad w.r.t to word embeddings

        # Fix for Issue #601

        # Check if gradient has shape [max_sequence,1,_] ( when model input in transpose of input sequence)

        if emb_grads[0].shape[1] == 1:
            grad = torch.transpose(emb_grads[0], 0, 1)[0].cpu().numpy()
        else:
            # gradient has shape [1,max_sequence,_]
            grad = emb_grads[0][0].cpu().numpy()

        embedding_layer.weight.requires_grad = original_state
        emb_hook.remove()
        self.model.eval()

        output = {"ids": ids[0].tolist(), "gradient": grad}

        return output

    def _tokenize(self, inputs):
        """Helper method that for `tokenize`
        Args:
            inputs (list[str]): list of input strings
        Returns:
            tokens (list[list[str]]): List of list of tokens as strings
        """
        return [self.tokenizer.convert_ids_to_tokens(self.tokenizer(x)) for x in inputs]

# Dataset Rotten Tomatoes para Teste de Modelos de Análise de Sentimentos

## Visão Geral
O **Rotten Tomatoes** é um dataset clássico para avaliação de modelos de análise de sentimentos, composto por resenhas de filmes classificadas como:
- **Positivas** (1)
- **Negativas** (0)

## Características Principais

### Composição
- Total de resenhas: ~10.000
- Distribuição balanceada entre classes
- Textos curtos (frases ou parágrafos breves)

In [22]:
dataset = HuggingFaceDataset("rotten_tomatoes", None, "test", shuffle=True)

README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

textattack: Loading datasets dataset rotten_tomatoes, split test.


In [23]:
#Exemplo
dataset[0:5]

[(OrderedDict([('text',
                "the film has the high-buffed gloss and high-octane jolts you expect of de palma , but what makes it transporting is that it's also one of the smartest , most pleasurable expressions of pure movie love to come from an american director in years .")]),
  1),
 (OrderedDict([('text',
                'spy-vs . -spy action flick with antonio banderas and lucy liu never comes together .')]),
  0),
 (OrderedDict([('text',
                "tsai ming-liang's witty , wistful new film , what time is it there ? , is a temporal inquiry that shoulders its philosophical burden lightly .")]),
  1),
 (OrderedDict([('text',
                'harvard man is a semi-throwback , a reminiscence without nostalgia or sentimentality .')]),
  0),
 (OrderedDict([('text',
                "it sounds like another clever if pointless excursion into the abyss , and that's more or less how it plays out .")]),
  0)]

# Ataque [HotFlip](https://arxiv.org/abs/1712.06751) a Modelos de NLP

## Visão Geral
O **HotFlip** é uma técnica de ataque adversarial proposta em 2017 que engana modelos de NLP através de perturbações mínimas no texto de entrada. O método utiliza gradientes do modelo para identificar e modificar palavras estratégicas, mantendo a legibilidade enquanto altera a classificação.

## Mecanismo do Ataque
Opera através de três ações principais:

1. **Substituição**  

2. **Inserção**  

3. **Exclusão**  

## Fluxo de Execução

### 1. Cálculo dos Gradientes
- Processa o texto no modelo
- Identifica tokens mais influentes via gradientes

### 2. Seleção de Alvos
- Rankeia palavras por impacto na decisão
- Prioriza termos com altos gradientes

### 3. Geração do Ataque
- Aplica uma das operações:
  - Substituição
  - Inserção
  - Exclusão

### 4. Validação
- Verifica:
  - Sucesso em enganar o modelo
  - Manutenção da coerência textual

## Requisitos Técnicos
- Acesso ao modelo (arquitetura + pesos)
- Capacidade de extrair gradientes
- Compatível com qualquer dataset do mesmo domínio

In [ ]:
attack = HotFlipEbrahimi2017.build(model_wrapped)
random_seed = 756

attack_args = AttackArgs(
    num_examples=100, # Specify the number of examples you want to attack
    log_to_csv="hotflip_attack_lstm", # Log results to a CSV
    disable_stdout=False,
    parallel=False,
    random_seed=random_seed,
    enable_advance_metrics=False,
)

attacker = Attacker(attack, dataset, attack_args)

from datetime import datetime

current_time = datetime.now().strftime("%Y-%m-%d-%H-%M")
attacker.attack_dataset()

textattack: Unknown if model of class <class '__main__.LSTMForClassification'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.
textattack: Logging to CSV at path hotflip_attack_lstm/2026-11-10-01-10-log.csv


Attack(
  (search_method): BeamSearch(
    (beam_width):  10
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapGradientBased(
    (top_n):  1
  )
  (constraints): 
    (0): MaxWordsPerturbed(
        (max_num_words):  2
        (compare_against_original):  True
      )
    (1): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.8
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
    (2): PartOfSpeech(
        (tagger_type):  nltk
        (tagset):  universal
        (allow_verb_noun_swap):  True
        (compare_against_original):  True
      )
    (3): RepeatModification
    (4): StopwordModification
  (is_black_box):  False
) 



  0%|          | 0/100 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py:1874: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)
[Succeeded / Failed / Skipped / Total] 0 / 1 / 1 / 2:   2%|▏         | 2/100 [00:02<02:09,  1.32s/it]

--------------------------------------------- Result 1 ---------------------------------------------
[[Positive (100%)]] --> [[[FAILED]]]

the film has the high-buffed gloss and high-octane jolts you expect of de palma , but what makes it transporting is that it's also one of the smartest , most pleasurable expressions of pure movie love to come from an american director in years .


--------------------------------------------- Result 2 ---------------------------------------------
[[Positive (100%)]] --> [[[SKIPPED]]]

spy-vs . -spy action flick with antonio banderas and lucy liu never comes together .




[Succeeded / Failed / Skipped / Total] 0 / 2 / 1 / 3:   3%|▎         | 3/100 [00:04<02:12,  1.36s/it]

--------------------------------------------- Result 3 ---------------------------------------------
[[Positive (100%)]] --> [[[FAILED]]]

tsai ming-liang's witty , wistful new film , what time is it there ? , is a temporal inquiry that shoulders its philosophical burden lightly .




... (152 outputs left out)

[Succeeded / Failed / Skipped / Total] 8 / 71 / 19 / 98:  98%|█████████▊| 98/100 [01:55<00:02,  1.18s/it]

--------------------------------------------- Result 98 ---------------------------------------------
[[Negative (96%)]] --> [[[FAILED]]]

after several scenes of this tacky nonsense , you'll be wistful for the testosterone-charged wizardry of jerry bruckheimer productions , especially because half past dead is like the rock on a wal-mart budget .




[Succeeded / Failed / Skipped / Total] 8 / 72 / 20 / 100: 100%|██████████| 100/100 [01:56<00:00,  1.17s/it]

--------------------------------------------- Result 99 ---------------------------------------------
[[Negative (100%)]] --> [[[FAILED]]]

dull , a road-trip movie that's surprisingly short of both adventure and song .


--------------------------------------------- Result 100 ---------------------------------------------
[[Positive (84%)]] --> [[[SKIPPED]]]

the hanukkah spirit seems fried in pork .



+-------------------------------+-------+
| Attack Results                |       |
+-------------------------------+-------+
| Number of successful attacks: | 8     |
| Number of failed attacks:     | 72    |
| Number of skipped attacks:    | 20    |
| Original accuracy:            | 80.0% |
| Accuracy under attack:        | 72.0% |
| Attack success rate:          | 10.0% |
| Average perturbed word %:     | 5.03% |
| Average num. words per input: | 18.71 |
| Avg num queries:              | 1.16  |
+-------------------------------+-------+


In [34]:

path = f'/content/hotflip_attack_lstm/{current_time}-log.csv'
df = pd.read_csv(path)
df.head()

,original_text,perturbed_text,original_score,perturbed_score,original_output,perturbed_output,ground_truth_output,num_queries,result_type
0,the film has the high-buffed gloss and high-oc...,the film has the high-buffed gloss and high-oc...,0.000000,0.000000,1,1,1,1,Failed
1,spy-vs . -spy action flick with antonio bander...,spy-vs . -spy action flick with antonio bander...,0.999998,0.999998,1,1,0,1,Skipped
2,"tsai ming-liang's witty , wistful new film , w...","tsai ming-liang's witty , wistful new film , w...",0.000000,0.000000,1,1,1,1,Failed
3,"harvard man is a semi-throwback , a reminiscen...","harvard man is a semi-throwback , a reminiscen...",0.248240,0.248240,0,0,0,1,Failed
4,it sounds like another clever if pointless exc...,it sounds like another clever if pointless exc...,0.000185,0.000185,0,0,0,1,Failed


In [35]:
df.loc[df.result_type=='Successful'][['original_text', 'perturbed_text']]

,original_text,perturbed_text
19,this is a fascinating film because there is no...,this is a fascinating film because there is no...
25,[breheny's] [[lensing]] of the new zealand and...,[breheny's] [[lesbians]] of the new zealand an...
26,a wishy-washy melodramatic movie that shows us...,a wishy-washy melodramatic movie that shows us...
33,serious [[movie-goers]] embarking upon this jo...,serious [[lesbians]] embarking upon this journ...
35,[[kapur]] fails to give his audience a single ...,[[attended]] fails to give his audience a sing...
60,. it's a testament to the film's considerable ...,. it's a testament to the film's considerable ...
67,highlighted by a gritty style and an excellent...,highlighted by a gritty style and an excellent...
95,not a [[schlocky]] creature feature but someth...,not a [[lesbians]] creature feature but someth...


# Análise do Resultado

| Attack Results                |       |
|-------------------------------|-------|
| Number of successful attacks  | 3     |
| Number of failed attacks      | 9     |
| Number of skipped attacks     | 8     |
| Original accuracy             | 60.0% |
| Accuracy under attack         | 45.0% |
| Attack success rate           | 25.0% |
| Average perturbed word %      | 5.21% |
| Average num. words per input  | 17.5  |
| Avg num queries               | 1.33  |
| Average Original Perplexity   | 77.74 |
| Average Attack Perplexity     | 94.25 |
| Average Attack USE Score      | 0.88  |

### Análise de Perplexidade
- **Original (77.74) → Ataque (94.25)**:  
  O aumento de 21.3% na perplexidade indica que os textos modificados ficaram um pouco menos "naturais" para o modelo, porém valores abaixo de 100 ainda sugerem saídas relativamente coerentes. Esse equilíbrio entre efetividade e naturalidade é crucial para ataques discretos.

### Similaridade Semântica (USE Score)
- **Pontuação USE 0.88**:  
  Indica forte preservação do significado original (onde 1.0=idêntico). Isso confirma que os ataques mantêm a intenção original enquanto enganam o modelo - um requisito essencial para exemplos adversariais válidos.

### Análise de Perturbações
- **5.21% de palavras alteradas**:  
  Essa baixa taxa de modificação é importante porque:
  1. Mostra a sensibilidade do modelo a mudanças mínimas
  2. Indica ataques eficientes (alto impacto por edição)
  3. Sugere que as vulnerabilidades não vêm de distorções grosseiras

## Interpretação dos Resultados
- **Taxa de sucesso de 25%**: Ainda que modesta, a combinação de:
  - Baixa degradação de perplexidade (94.25)
  - Alta pontuação USE (0.88)
  - Perturbações mínimas (5.21%)
  
  Revela que:
  - O modelo falha em casos adversariais semanticamente próximos
  - Os ataques são computacionalmente eficientes (1.33 consultas)
  - Queda de 15% na acurácia (60%→45%) confirma vulnerabilidade relevante

# Atividade: Reprodução de Ataque Adversarial em Texto em Modelos de NLP

Visto que não há hiperparâmetros para variação nesta implementação do HotFlip, foi feita uma reprodução do ataque com uma amostra de 100 exemplos do dataset Rotten Tomatoes, utilizando o modelo LSTM treinado anteriormente. O objetivo é observar a eficácia do ataque e analisar métricas como taxa de sucesso, perplexidade e similaridade semântica.



| Métrica | Valor Obtido |
| :--- | :--- |
| **Ataques com Sucesso (Successful)** | 8 |
| **Ataques Falhos (Failed)** | 72 |
| **Ataques Ignorados (Skipped)** | 20 |
| **Acurácia Original do Modelo** | 80.0% |
| **Acurácia sob Ataque** | 72.0% |
| **Taxa de Sucesso do Ataque (ASR)** | 10.0% |
| **Percentual Médio de Palavras Perturbadas** | 5.03% |
| **Média de Palavras por Entrada** | 18.71 |
| **Média de Consultas (Queries por Exemplo)** | 1.16 |

### Desempenho e Eficácia do Ataque
- **Acurácia Original (80.0%) vs. Sob Ataque (72.0%)**:
  O ataque causou uma redução absoluta de 8 pontos percentuais (ou 10% relativo) na acurácia da rede LSTM. Das 80 instâncias que o modelo havia classificado corretamente (as 20 incorretas foram automaticamente marcadas como *Skipped*), o HotFlip conseguiu forçar a inversão de rótulo em 8 exemplos.
- **Taxa de Sucesso (ASR de 10.0%)**:
  A taxa de sucesso pode parecer modesta à primeira vista, mas reflete o rigor das restrições implementadas na receita original de Ebrahimi et al. (2017):
  1. Orçamento máximo de apenas **2 palavras alteradas** (`max_num_words=2`).
  2. Limiar rígido de similaridade por cosseno nos embeddings ($\ge 0.8$).
  3. Exigência de preservação da mesma classe gramatical (*POS Tagging*) e proibição de alteração de *stopwords*.
  Essas barreiras priorizam a plausibilidade do texto e a imperceptibilidade humana em detrimento da agressividade bruta do ataque.


### Perturbação Mínima e Imperceptibilidade
- **Apenas 5.03% de palavras alteradas**:
  Considerando uma extensão média de ~19 palavras por resenha, a alteração de apenas ~1 palavra por sentença foi suficiente nos casos bem-sucedidos para induzir o modelo ao erro. 
- Isso evidencia a vulnerabilidade pontual de redes neurais recorrentes (LSTM): o modelo atribui peso desproporcional a determinados termos-chave (como demonstrado na substituição de qualificadores e substantivos), permitindo que perturbações mínimas no espaço latente desloquem a representação além da fronteira de decisão.


### Eficiência Computacional (Vantagem White-Box)
- **Média de 1.16 consultas por exemplo**: Enquanto ataques *black-box* (baseados em pontuação ou decisão) frequentemente demandam dezenas ou centenas de consultas para explorar heurísticas de substituição, o HotFlip utiliza o **gradiente direcional** da perda em relação à representação *one-hot* de entrada ($\nabla_x \mathcal{L}$).
- Essa aproximação de primeira ordem aponta diretamente qual substituição no vocabulário maximiza a perda com custo computacional mínimo, realizando o ataque de forma quase instantânea, aproveitando a vantagem de acesso total ao modelo (*white-box*).


### Conclusão 
A reprodução confirmou que o HotFlip é um ataque de alta eficiência computacional e perturbação cirúrgica. Embora as restrições gramaticais estritas limitem a taxa de sucesso, os exemplos gerados comprovam que representações contínuas de texto ainda são suscetíveis a manipulações imperceptíveis quando o atacante possui acesso total aos gradientes da arquitetura vítima.